[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mento-calc/mento/blob/main/docs/source/examples/viga_rectangular_verificacion_CIRSOC_201-25.ipynb)

In [ ]:
# Google Colab no trae mento instalado, así que lo instalamos sólo cuando el
# notebook corre allí. La guarda evita que la documentación llame a pip.
import sys

if "google.colab" in sys.modules:
    %pip install mento

# Verificación de viga rectangular — CIRSOC 201-25

Acá la armadura ya está definida — porque viene de un plano, de un cálculo anterior o de una
estructura existente — y lo que queremos saber es si verifica según el **CIRSOC 201-2025**.

Es el camino inverso al del ejemplo de
[diseño](viga_rectangular_diseno_CIRSOC_201-25.ipynb): en lugar de pedirle a mento que elija
las barras, se las imponemos y le pedimos la relación demanda-capacidad.


#### Idioma español para salida de anexos

In [ ]:
import mento

mento.set_language("es")

**Materiales y geometría de la sección**

In [ ]:
from mento import Concrete_CIRSOC_201_25, SteelBar, RectangularBeam, cm, mm, kN, MPa, kNm
from mento import Forces, Node

hormigon = Concrete_CIRSOC_201_25(name="H-25", f_c=25 * MPa)
acero = SteelBar(name="ADN 420", f_y=420 * MPa)

viga = RectangularBeam(
    label="V201",
    concrete=hormigon,
    steel_bar=acero,
    width=20 * cm,
    height=50 * cm,
    c_c=2.5 * cm,
)
viga.data

**Armadura existente**

La armadura longitudinal se carga por cara y por capa; los estribos, con la cantidad de
estribos, el diámetro y la separación longitudinal.

In [ ]:
# Armadura longitudinal inferior: 2Ø16 en la primera capa y 1Ø12 en la segunda
viga.set_longitudinal_rebar_bot(n1=2, d_b1=16 * mm, n2=1, d_b2=12 * mm)

# Armadura longitudinal superior: 2Ø12
viga.set_longitudinal_rebar_top(n1=2, d_b1=12 * mm)

# Estribos: 1 estribo (2 ramas) Ø8 cada 20 cm
viga.set_transverse_rebar(n_stirrups=1, d_b=8 * mm, s_l=20 * cm)

viga.plot()

**Estados de carga y nodo**

In [ ]:
f1 = Forces(label="1.2D+1.6L", M_y=80 * kNm, V_z=50 * kN)
f2 = Forces(label="1.2D+1.6L+W", M_y=-30 * kNm, V_z=55 * kN)

nodo = Node(section=viga, forces=[f1, f2])
nodo

**Verificación**

`check()` corre flexión y corte para todas las combinaciones. La columna DCR es la relación
demanda-capacidad: verifica mientras sea menor o igual a 1.

In [ ]:
nodo.check()
nodo.results

In [ ]:
nodo.check_flexure()

In [ ]:
nodo.check_shear()

**Lectura de la verificación desde el código**

Después de un `check()`, `flexure_design` y `shear_design` describen la armadura verificada y
su DCR, sin tener que leer la tabla. Los DCR son la envolvente de todos los estados de carga,
es decir, el de la combinación que gobierna cada cara.

In [ ]:
flexion = viga.flexure_design
corte = viga.shear_design

print("Armadura inferior:", flexion.bottom, f"→ As = {flexion.bottom.A_s.to('cm**2'):.2f~P}")
print("Armadura superior:", flexion.top, f"→ As = {flexion.top.A_s.to('cm**2'):.2f~P}")
print("Estribos:", corte)
print(f"DCR flexión = {flexion.DCR:.2f} | DCR corte = {corte.DCR:.2f}")
print("Verifica" if max(flexion.DCR, corte.DCR) <= 1 else "No verifica")

**Memoria de cálculo detallada**

Para la combinación que gobierna, con cada verificación del reglamento paso a paso.

In [ ]:
nodo.flexure_results_detailed()

In [ ]:
nodo.shear_results_detailed()